In [ ]:
import re
import threading
from concurrent.futures import ThreadPoolExecutor
from datetime import datetime
from pathlib import Path
from urllib.parse import quote

import pandas as pd
import pypdf
import pytesseract
from pdf2image import convert_from_path, pdfinfo_from_path
from shared_config.sqlserver import PROD_SERVER, PYRO_DATABASE
from shared_utils.sqlserver import get_engine, replace_table

In [ ]:
# Toggles
test = False  # set to True to use test PDFs; set to False to use production PDFs

In [ ]:
PROJECT_PATH = r'C:\analytics-work\development\extract_material_cert_acce_rr'

PROD_PDF_PATH = rf'{PROJECT_PATH}\data'

TABLE_NAME = 'material_cert_acce_rr'

# Leave blank if the PDFs have no SharePoint home; pdf_url is then omitted.
SHAREPOINT_URL = ''

POPPLER_PATH = r'C:\Program Files\poppler\Library\bin'
TESSERACT_PATH = r'C:\Program Files\Tesseract-OCR\tesseract.exe'

LOW_DPI = 160
HIGH_DPI = 275
OCR_CONFIG = r'--oem 3 --psm 6'
MAX_WORKERS_PDF = 4
MAX_WORKERS_PAGE = 1

# Pages with fewer characters than this in the PDF text layer are treated as scans.
MIN_TEXT_CHARS = 80

# Plausible value ranges. Parsing is by range, not column position, because
# OCR inserts stray characters and shifts columns.
RC_RANGE = (30, 50)
AWT_RANGE = (270, 300)
PAW_RANGE = (440, 490)
TG_RANGE = (150, 400)
AVG_TOLERANCE = 1.0  # allowed gap between computed and printed spool averages

PDF_PATH = PROD_PDF_PATH

In [ ]:
prod_engine = get_engine(PROD_SERVER, PYRO_DATABASE)

In [ ]:
# SharePoint url
def file_to_sharepoint_url(file, pdf_path, sharepoint_url):
    """
    Build the SharePoint URL for a PDF from its local path under pdf_path.

    Assumes pdf_path is a local OneDrive shortcut to the SharePoint folder at
    sharepoint_url -- the two must point to the same folder level so the
    relative paths line up.
    """
    rel_path = Path(file).resolve().relative_to(Path(pdf_path).resolve())
    rel_url = quote(str(rel_path).replace('\\', '/'))
    return f'{sharepoint_url}/{rel_url}'

In [ ]:
# OCR helpers
pytesseract.pytesseract.tesseract_cmd = TESSERACT_PATH

In [ ]:
def _ocr_page(file, page_num, dpi, config_str=OCR_CONFIG):
    """Render one 1-indexed page to an image and OCR it. Returns (page_num, text)."""
    imgs = convert_from_path(
        file,
        dpi=dpi,
        poppler_path=POPPLER_PATH,
        first_page=page_num,
        last_page=page_num,
    )
    if not imgs:
        return page_num, ''
    return page_num, pytesseract.image_to_string(imgs[0], config=config_str)

In [ ]:
def _ocr_many(file, page_numbers, dpi, config_str=OCR_CONFIG):
    """OCR a set of pages in parallel. Returns {page_num: text}."""
    page_numbers = sorted(set(page_numbers))
    if not page_numbers:
        return {}
    with ThreadPoolExecutor(max_workers=MAX_WORKERS_PAGE) as ex:
        return dict(ex.map(lambda p: _ocr_page(file, p, dpi, config_str), page_numbers))

In [ ]:
def _text_layer(file):
    """Return {page_num: text} from the PDF text layer ('' for scanned pages)."""
    try:
        reader = pypdf.PdfReader(file)
        texts = {}
        for pn, page in enumerate(reader.pages, 1):
            text = page.extract_text() or ''
            texts[pn] = text if len(text.strip()) >= MIN_TEXT_CHARS else ''
        return texts
    except Exception as e:
        print(f'    pypdf could not open file: {e}; falling back to OCR scan.')
        return {}

In [ ]:
def read_relevant_pages(file, dpi_value=LOW_DPI):
    """
    Two-pass read of one cert PDF.

    Pass 1: classify every page, from the text layer where there is one and from a
    low-DPI OCR otherwise (certs are often a mix of both).
    Pass 2: re-OCR only the relevant scanned pages at HIGH_DPI.

    Returns {page_num: (kinds, text)} for relevant pages only.
    """
    n_pages = pdfinfo_from_path(file, poppler_path=POPPLER_PATH)['Pages']
    layer = _text_layer(file)

    scan_pages = [p for p in range(1, n_pages + 1) if not layer.get(p)]
    low = _ocr_many(file, scan_pages, dpi_value)

    texts = {p: layer.get(p) or low.get(p, '') for p in range(1, n_pages + 1)}
    kinds = {p: classify_page_text(t) for p, t in texts.items()}
    relevant = {p: k for p, k in kinds.items() if k}

    rescan = [p for p in relevant if not layer.get(p)]
    high = _ocr_many(file, rescan, HIGH_DPI)

    return {p: (relevant[p], high.get(p) or texts[p]) for p in relevant}

In [ ]:
# Patterns and parsing helpers
MONTHS = (
    'January|February|March|April|May|June|July|August|September|'
    'October|November|December'
)

# Lot numbers look like 92M0114922. OCR may put a space inside (92M01 14922) or
# read a zero as the letter O.
LOT_RE = re.compile(r'\d{2}M[0O\d](?:\s?\d){6}')
PART_RE = re.compile(r'(MSRR\d{4}-[A-Z0-9]+)')
MFG_DATE_RE = re.compile(
    rf'Date\s*of\s*Manufacture\s*:?\s*(\d{{1,2}}\s+(?:{MONTHS})\s+\d{{4}})', re.I
)
TG_LABEL_RE = re.compile(r't[g9q]\s*by\s*dma\s*in', re.I)
TG_VALUE_RE = re.compile(r'(?<!\d)(\d{3}\.\d)')
TACK_LINE_RE = re.compile(r'tack\s*force\s*\(', re.I)
TACK_AVG_RE = re.compile(r'tack\s*force\s*avg\W+\w{2,4}\W+(\d+[.,]\d+)', re.I)
TACK_READING_RE = re.compile(r'(\d+[.,]\d{3})')
PANEL_RE = re.compile(r'(\d{3})/(\d{3})')
AVG_NUM_RE = re.compile(r'\d{2,3}\.\d')
SPOOL_LOT_HEADER_RE = re.compile(r'lot\s*averages\s*report\s*for\s*lot\s*:?\s*(\S+)', re.I)

In [ ]:
def clean_lot(raw):
    """Normalize an OCR'd lot number: drop spaces, letter O -> zero."""
    s = re.sub(r'\s+', '', raw).upper()
    return s[:3] + s[3:].replace('O', '0')

In [ ]:
def find_lot(text):
    """First lot number found in a block of text, or None."""
    m = LOT_RE.search(text)
    return clean_lot(m.group(0)) if m else None

In [ ]:
def lot_from_filename(filename):
    return find_lot(filename)

In [ ]:
def classify_page_text(text):
    """Return the set of record kinds a page holds: lot_info, spool, tg, tack."""
    low = re.sub(r'\s+', ' ', (text or '').lower())
    kinds = set()
    if 'date of manufacture' in low and 'lot no' in low:
        kinds.add('lot_info')
    if 'lot averages report' in low:
        kinds.add('spool')
    if TG_LABEL_RE.search(low):
        kinds.add('tg')
    if 'tack force' in low:
        kinds.add('tack')
    return kinds

In [ ]:
def page_lot(text, kinds, filename=''):
    """Lot number a page belongs to (a PDF can hold several lots)."""
    m = SPOOL_LOT_HEADER_RE.search(text)
    if m and 'spool' in kinds:
        lot = find_lot(m.group(1)) or clean_lot(m.group(1))
        if lot:
            return lot
    m = re.search(r'lot\s*(?:no|number)\s*:?\s*(' + LOT_RE.pattern + ')', text, re.I)
    if m:
        return clean_lot(m.group(1))
    return find_lot(text) or lot_from_filename(filename)

In [ ]:
def extract_lot_info(text):
    """part_number and mfg_date from a certificate-of-analysis cover page."""
    part = PART_RE.search(text)
    date = MFG_DATE_RE.search(text)
    mfg_date = None
    if date:
        mfg_date = datetime.strptime(re.sub(r'\s+', ' ', date.group(1)).title(), '%d %B %Y')
    return {'part_number': part.group(1) if part else None, 'mfg_date': mfg_date}

In [ ]:
def _to_float(s):
    try:
        return float(s.replace(',', '.'))
    except ValueError:
        return None

In [ ]:
def _in(v, rng):
    return v is not None and rng[0] <= v <= rng[1]

In [ ]:
def extract_tg(text):
    """Per-panel Tg by DMA (deg C). Returns [(panel, value), ...] in page order."""
    out = []
    for line in text.splitlines():
        m = TG_LABEL_RE.search(line)
        if not m or re.search(r'\bavg\b', line, re.I):
            continue
        val = next(
            (float(v) for v in TG_VALUE_RE.findall(line[m.start():]) if _in(float(v), TG_RANGE)),
            None,
        )
        if val is None:
            continue
        panel = PANEL_RE.search(line)
        out.append((panel.group(2) if panel else None, val))
    return out

In [ ]:
def extract_tack(text):
    """
    ASTM D8336 Nottingham tack, per panel. Returns [(readings, avg), ...].

    A readings line ('Tack Force (lbf) a - b - c') is followed by its
    'Tack Force Avg (lbf)' line. The printed average is used; if it is missing
    the mean of the readings is used instead.
    """
    out, readings = [], None
    for line in text.splitlines():
        avg = TACK_AVG_RE.search(line)
        if avg:
            val = _to_float(avg.group(1))
            if readings and val is None:
                val = round(sum(readings) / len(readings), 3)
            out.append((readings or [], val))
            readings = None
            continue
        if TACK_LINE_RE.search(line):
            after = line[TACK_LINE_RE.search(line).end():]
            readings = [_to_float(v) for v in TACK_READING_RE.findall(after)]
            readings = [v for v in readings if v is not None]
    return out

In [ ]:
def repair_paw(tokens):
    """PAW is garbled by an inserted or substituted digit (4162, 160 -> 46x).
    Only used when no token is already in range. Returns (value, note) or (None, None)."""
    for tok in tokens:
        if not tok.isdigit():
            continue
        if len(tok) == 4:  # extra digit inserted
            fixes = {int(tok[:i] + tok[i + 1:]) for i in range(4)}
            fixes = {f for f in fixes if _in(f, PAW_RANGE)}
            if len(fixes) == 1:
                return fixes.pop(), f'OCR read {tok}: dropped extra digit'
        if len(tok) == 3 and tok[0] in '1l7':  # leading 4 misread
            fixed = int('4' + tok[1:])
            if _in(fixed, PAW_RANGE):
                return fixed, f'OCR read {tok}: first digit set to 4'
    return None, None

In [ ]:
def _split_glued(tokens):
    """Some scans glue PAW and RC into one token ('45939' -> PAW 459, RC 39)."""
    for tok in tokens:
        if re.fullmatch(r'\d{5}', tok):
            paw, rc = int(tok[:3]), int(tok[3:])
            if _in(paw, PAW_RANGE) and _in(rc, RC_RANGE):
                return rc, paw
    return None, None

In [ ]:
def extract_spools(text, lot):
    """
    Spool rows from a 'Lot Averages Report' page.
    Returns (rows, printed_averages) where rows are dicts of spool_number, rc, paw, note
    and printed_averages is {'rc':..., 'paw':...} (either may be None) or None.
    """
    rows, averages = [], None
    for line in text.splitlines():
        if re.search(r'average', line, re.I):
            vals = [float(v) for v in AVG_NUM_RE.findall(line)]
            found = {
                'rc': next((v for v in vals if _in(v, RC_RANGE)), None),
                'paw': next((v for v in vals if _in(v, PAW_RANGE)), None),
            }
            if found['rc'] is not None or found['paw'] is not None:
                averages = found
            continue

        tokens = line.split()
        if not tokens:
            continue
        first = re.sub(r'[^0-9A-Za-z]', '', tokens[0])
        if not (
            len(first) in (12, 13)
            and first[-3:].isdigit()
            and re.fullmatch(r'\d{2}[A-Za-z0]?\d{7}|\d{2}[A-Za-z]\d{7}', first[:-3])
        ):
            continue

        rest = [re.sub(r'[^0-9.]', '', t) for t in tokens[1:]]
        rest = [t for t in rest if t and t != '.']
        nums = [float(t) for t in rest if re.fullmatch(r'\d+(\.\d+)?', t)]

        rc = next((v for v in nums if _in(v, RC_RANGE)), None)
        paw = next((v for v in nums if _in(v, PAW_RANGE)), None)
        notes = []
        if paw is None:
            paw, note = repair_paw(rest)
            if note:
                notes.append(note)
        if rc is None or paw is None:
            g_rc, g_paw = _split_glued(rest)
            if g_rc is not None:
                rc = rc if rc is not None else g_rc
                if paw is None:
                    paw = g_paw

        spool_no = first[-3:]
        rows.append({
            'spool_number': f'{lot}{spool_no}' if lot else first,
            'rc': rc,
            'paw': paw,
            'note': '; '.join(notes) or None,
        })
    return rows, averages

In [ ]:
def check_averages(rows, averages):
    """True if computed RC/PAW means match the printed 'Averages:' line."""
    if not averages or not rows:
        return None
    checked = False
    for col in ('rc', 'paw'):
        if averages.get(col) is None:  # printed average was unreadable
            continue
        vals = [r[col] for r in rows if r[col] is not None]
        if not vals or abs(sum(vals) / len(vals) - averages[col]) > AVG_TOLERANCE:
            return False
        checked = True
    return True if checked else None

In [ ]:
# Per-file orchestration
def _lot_record():
    return {
        'part_number': None,
        'mfg_date': None,
        'spools': [],
        'tg': [],
        'tack': [],
        'spool_averages': None,
    }

In [ ]:
def _dedupe(items):
    """Drop exact repeats (a page can appear twice in a PDF), keeping order."""
    seen, out = set(), []
    for item in items:
        key = repr(item)
        if key not in seen:
            seen.add(key)
            out.append(item)
    return out

In [ ]:
def _process_file(file, pdf_path, sharepoint_url, log):
    """Two-pass read of one cert PDF. Returns a DataFrame of per-spool rows.

    A PDF can hold more than one lot, so everything is grouped by lot first.
    """
    log(f'\n  Processing {Path(file).name} ...')

    pages = read_relevant_pages(file, LOW_DPI)
    if not any({'tg', 'tack', 'spool'} & kinds for kinds, _ in pages.values()):
        retry = LOW_DPI + 50
        log(f'    Nothing relevant at {LOW_DPI} DPI, retrying at {retry} DPI...')
        pages = read_relevant_pages(file, retry)
        if not pages:
            log(f'    Still none; skipping {Path(file).name}.')
            return pd.DataFrame()

    lots: dict[str, dict] = {}
    for pn in sorted(pages):
        kinds, text = pages[pn]
        lot = page_lot(text, kinds, Path(file).name)
        if lot is None:
            log(f'    Page {pn}: could not determine lot; skipped.')
            continue
        rec = lots.setdefault(lot, _lot_record())

        if 'lot_info' in kinds:
            info = extract_lot_info(text)
            rec['part_number'] = rec['part_number'] or info['part_number']
            rec['mfg_date'] = rec['mfg_date'] or info['mfg_date']
        if 'tg' in kinds:
            rec['tg'].extend(extract_tg(text))
        if 'tack' in kinds:
            rec['tack'].extend(extract_tack(text))
        if 'spool' in kinds:
            rows, averages = extract_spools(text, lot)
            rec['spools'].extend(rows)
            if averages:
                rec['spool_averages'] = averages

    out = []
    for lot, rec in sorted(lots.items()):
        spools = _dedupe(rec['spools'])
        tg = [v for _, v in _dedupe(rec['tg'])]
        tack = [avg for _, avg in _dedupe(rec['tack']) if avg is not None]
        avg_ok = check_averages(spools, rec['spool_averages'])

        lot_fields = {
            'part_number': rec['part_number'],
            'lot_number': lot,
            'mfg_date': rec['mfg_date'],
        }
        for i, v in enumerate(tg, 1):
            lot_fields[f'tg_{i}'] = v
        lot_fields['tg_avg'] = round(sum(tg) / len(tg), 1) if tg else None
        for i, v in enumerate(tack, 1):
            lot_fields[f'tack_{i}'] = v
        lot_fields['tack_avg'] = round(sum(tack) / len(tack), 3) if tack else None

        if not spools:
            spools = [
                {'spool_number': None, 'rc': None, 'paw': None, 'note': 'no spool table found'}
            ]

        for s in spools:
            notes = [s['note']] if s['note'] else []
            if avg_ok is False:
                notes.append('lot RC/PAW average does not match printed averages')
            if s['spool_number'] and (s['rc'] is None or s['paw'] is None):
                notes.append('RC or PAW not read')
            out.append({
                **lot_fields,
                'spool_number': s['spool_number'],
                'rc': s['rc'],
                'paw': s['paw'],
                'needs_review': bool(notes),
                'notes': '; '.join(notes) or None,
                'pdf_path': str(Path(file).resolve()),
                **(
                    {'pdf_url': file_to_sharepoint_url(file, pdf_path, sharepoint_url)}
                    if sharepoint_url
                    else {}
                ),
            })

    df = pd.DataFrame(out)
    if not df.empty:
        log(f'    Extracted {len(df)} spool record(s) from {df["lot_number"].nunique()} lot(s).')
    return df


_print_lock = threading.Lock()

In [ ]:
def process_file(file, pdf_path, sharepoint_url=''):
    """Thread-safe wrapper: buffers this file's messages and prints them together,
    so output from parallel files does not interleave."""
    msgs = []
    try:
        return _process_file(file, pdf_path, sharepoint_url, msgs.append)
    finally:
        with _print_lock:
            print('\n'.join(msgs))

In [ ]:
# Run over the PDFs
def run(pdf_path, sharepoint_url):
    """OCR every PDF under pdf_path. Always processes all files; the SQL table is
    replaced on each run, so there is no processed-file tracking."""
    root = Path(pdf_path)
    all_pdfs = [
        pdf for pdf in sorted(root.rglob('*')) if pdf.suffix.lower() == '.pdf'
    ]
    if not all_pdfs:
        print(f'No PDFs found in {pdf_path}')
        return None

    print(f'Found {len(all_pdfs)} PDF file(s) requiring processing...')

    results = []
    with ThreadPoolExecutor(max_workers=MAX_WORKERS_PDF) as executor:
        futures = {
            executor.submit(process_file, pdf, pdf_path, sharepoint_url): pdf
            for pdf in all_pdfs
        }
        for future in futures:
            try:
                df = future.result()
            except Exception as e:
                print(f'    FAILED {futures[future].name}: {e}')
                continue
            if not df.empty:
                results.append(df)

    df_all = pd.concat(results, ignore_index=True) if results else pd.DataFrame()

    print(
        f'\n{"=" * 64}\nTOTAL: {len(df_all)} records from {len(all_pdfs)} PDF(s)\n{"=" * 64}'
    )

    if not df_all.empty:
        df_all = df_all.sort_values(['lot_number', 'spool_number']).reset_index(drop=True)
        df_all['mfg_date'] = pd.to_datetime(df_all['mfg_date']).dt.normalize()

    return df_all

In [ ]:
df_all = run(PDF_PATH, SHAREPOINT_URL)

In [ ]:
if df_all is None or df_all.empty:
    print('No data extracted: skipping DB write')
else:
    replace_table(df_all, table_name=TABLE_NAME, engine_name=prod_engine)

In [ ]:
# Rows worth a manual look
if df_all is not None and not df_all.empty:
    df_all[df_all['needs_review']]